In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# ==========================================
# 1. CONFIGURATION
# ==========================================
FILE_PATH = 'airveda_data_aug 2025.csv'
WINDOW_SIZE = 12  # Look back 12 steps (e.g., 6 hours if data is 30-min interval)
TRAIN_SPLIT = 0.8 # 80% for training, 20% for testing

# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
# Load data
df = pd.read_csv(FILE_PATH)

# Parse Dates
df['CreatedDate'] = pd.to_datetime(df['CreatedDate'], format='%d-%m-%Y %H:%M')
df = df.sort_values('CreatedDate')
df.set_index('CreatedDate', inplace=True)

# Select Features and Target
# We use PM2.5, PM10, Temp, Humidity to predict AQI
feature_cols = ['PM2.5_1205250013', 'PM10_1205250013', 'Temperature_1205250013', 'Humidity_1205250013']
target_col = 'AQI_1205250013'

# Create a dataframe with features first, target last (for easier scaling)
data = df[feature_cols + [target_col]].values

# Normalize Data (LSTM works best with 0-1 scaling)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# ==========================================
# 3. CREATE SLIDING WINDOW DATASET
# ==========================================
# X: (samples, window_size, 4 features)
# y: (samples, 1 target)

def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        # Gather input features (columns 0 to 3) for the window
        a = dataset[i:(i + look_back), 0:4] 
        X.append(a)
        # Gather target (column 4, which is AQI) for the next step
        Y.append(dataset[i + look_back, 4]) 
    return np.array(X), np.array(Y)

X, y = create_dataset(scaled_data, WINDOW_SIZE)

# Split into Train and Test
train_size = int(len(X) * TRAIN_SPLIT)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"Training Shape: X={X_train.shape}, y={y_train.shape}")
print(f"Testing Shape:  X={X_test.shape}, y={y_test.shape}")

# ==========================================
# 4. BUILD MULTIVARIATE LSTM MODEL
# ==========================================
model = Sequential()

# LSTM Layer 1
model.add(LSTM(units=64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2)) # Prevent overfitting

# LSTM Layer 2
model.add(LSTM(units=32, return_sequences=False))
model.add(Dropout(0.2))

# Output Layer (1 unit for AQI)
model.add(Dense(units=1))

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
model.summary()

# ==========================================
# 5. TRAINING
# ==========================================
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# ==========================================
# 6. EVALUATION & VISUALIZATION
# ==========================================
# Make predictions
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Invert predictions to original scale (AQI)
# We need to create a dummy array to inverse transform because scaler expects 5 columns
def invert_predictions(predictions, original_scaled_data, scaler, look_back, start_idx):
    # Get the matching feature data to reconstruct the row structure
    # We take the features from the original scaled data corresponding to the prediction rows
    # Prediction index i corresponds to original data index i + look_back
    dummy_features = original_scaled_data[start_idx + look_back : start_idx + look_back + len(predictions), 0:4]
    
    # Concatenate features + predicted AQI
    combined = np.concatenate((dummy_features, predictions), axis=1)
    
    # Inverse transform
    inverted = scaler.inverse_transform(combined)
    
    # Return only the AQI column (last column)
    return inverted[:, 4]

# Invert Train and Test predictions
real_y_test = invert_predictions(y_test.reshape(-1, 1), scaled_data, scaler, WINDOW_SIZE, train_size)
pred_y_test = invert_predictions(test_predict, scaled_data, scaler, WINDOW_SIZE, train_size)

# Plot Results
plt.figure(figsize=(14, 6))
plt.plot(real_y_test, color='black', label='Actual AQI')
plt.plot(pred_y_test, color='green', label='Predicted AQI (Multivariate LSTM)')
plt.title('Air Quality Prediction (Test Set)')
plt.xlabel('Time Steps')
plt.ylabel('AQI')
plt.legend()
plt.show()

# Plot Loss
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'sklearn'